# Analysis of Post-Predictives

This aim of this notebook is to gain stronger intuitions regarding our model selection results. That is, *why* is CO the best model when fitting adaptation?

One idea I wish to test is how the simulated adaptation measures from each model compare to the actual adaptation measures. In other words, to correlate the two and visualize them through a scatterplot. We can add further detail by splitting the data by those that came from rotations of magnitude less than 6 deg and those that came from rotations of magnitudes greater than or equal to 6 deg (i.e., from the theoretical saturation zone). 

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import scipy.optimize
from scipy.optimize import minimize
from scipy.stats import norm
from scipy.special import logsumexp
import warnings
from scipy.optimize import curve_fit
from scipy.optimize import approx_fprime
import scipy.stats as stats
from pybads import BADS

from matplotlib.colors import LinearSegmentedColormap

# TEMP
warnings.simplefilter("ignore", category=FutureWarning)

from src.src import *

# Set some defaults
plt.rc("axes.spines", top=False, right=False)
plt.rcParams["mathtext.default"] = "regular"
sns.set_theme("paper", font_scale=1.3)
sns.set_style("ticks")

%config InlineBackend.figure_format = "retina"

---
## Helper functions

In [2]:
# Helper functions
def converter(input_str):
    return np.fromstring(input_str[1:-1], sep=' ')

---
## Import data

In addition to empirical adaptation metrics from each participant we want posterior predictives from at least CO and PIECE models. It may be easier to simulate posterior predictives within this notebook (see "analyze-models.ipynb"), as opposed to importing them. Due to the stochastic nature of the simulations, we want to see how things change upon repeated simulations. 


In [3]:
# Read in data 
df_behav = pd.read_csv("combined-data.csv")
fits_adapt = pd.read_csv("model-fits/fits_adapt.csv", converters={"params": converter})


In [4]:
# Choose a sample subject
subj_id = 1
sample_subj = df_behav.loc[(df_behav["SN"] == subj_id) & (df_behav["session"] == "adapt"), :].reset_index(drop=True)
sigma_motor = sample_subj.loc[0, "sigma_motor_adapt"]

# Pull out best-fit adaptation params for chosen subject:
co_mles = fits_adapt.loc[fits_adapt["model"] == "co", ["C", "B", "sigma_comb", "b"]]
piece_mles = fits_adapt.loc[fits_adapt["model"] == "piece", ["alpha", "beta", "sigma_p", "sigma_pert", "b"]]


In [5]:
# Simulate adaptation with best-fit parameters

# CO model
xhat_co = co_simulate_adapt(
    df=sample_subj,
    C=co_mles["C"].values[0], 
    B=co_mles["B"].values[0],
    sigma_comb=co_mles["sigma_comb"].values[0],
    b=co_mles["b"].values[0],
    sigma_plan=3
)
# Need to match number of simulated adapt measures without importing last bsl trial
xhat_co = np.insert(xhat_co, 0, np.random.normal(loc=co_mles["b"].values[0], scale=sigma_motor))  
adapt_co = xhat_co[2::2] - xhat_co[0:-1:2]

# PIECE model
xhat_piece = piece_simulate_adapt(
    df=sample_subj,
    alpha=piece_mles["alpha"].values[0], 
    beta=piece_mles["beta"].values[0],
    sigma_p=piece_mles["sigma_p"].values[0],
    sigma_pert=piece_mles["sigma_pert"].values[0],
    b=piece_mles["b"].values[0],
)
# Need to match number of simulated adapt measures without importing last bsl trial
xhat_piece = np.insert(xhat_piece, 0, np.random.normal(loc=piece_mles["b"].values[0], scale=sigma_motor))  
adapt_piece = xhat_piece[2::2] - xhat_piece[0:-1:2]

In [8]:
# Find perturbation trials in data frame
pert_indices = np.flatnonzero(sample_subj["pert_trial"])

# Trim subject data frame to only adaptation trials
df_trimmed = sample_subj.loc[pert_indices, :]
SN = df_trimmed.loc[:, "SN"]
rotation = df_trimmed.loc[:, "delta_xv"]

# Outlier indices
idx = df_trimmed["adaptation"].isna()

# Create data frame with all perturbation trials
df_plot = pd.DataFrame({
    "delta_xv" : df_trimmed.loc[:, "delta_xv"],
    "adaptation" : df_trimmed.loc[:, "adaptation"],
    "x_f" : df_trimmed.loc[:, "x_f_clean"],
    "xhat_co" : xhat_co[1::2],
    "adapt_co" : adapt_co,
    "xhat_piece" : xhat_piece[1::2], 
    "adapt_piece" : adapt_piece, 
    "rotation" : rotation,
    "SN" : SN
})


In [11]:
idx[0:11]

0     False
2     False
4     False
6     False
8     False
10    False
12    False
14    False
16    False
18    False
20     True
Name: adaptation, dtype: bool

In [7]:
print(df_plot["adaptation"].isna().sum())
print(df_plot["adapt_co"].isna().sum())

8
0


---
## Simulate behavior using $\theta_{\text{MLE}}$

Does the exact protocol matter? In theory, no, since we treat data as being i.i.d. However, may want to use same protocol, in which case we also need to be aware of which measures were removed due to being outliers. 

---
## Visualize real vs simulated with scatterplots

What we're looking for is breakdown of bias and variance in model predictions. For instance, CO and PIECE look pretty similar in our posterior predictive plots from the manuscript; therefore, we're looking to see if my intuitions are correct regarding the stronger linearity for CO compared to PIECE in response to smaller errors as well as whether PIECE is overpredicting the variability of responses. 

Should plot with dashed unity line. Compute MSE. 